In [1]:
%pip install -U -r requirements.txt

  Using cached aiohttp-3.11.11-cp310-cp310-macosx_10_9_x86_64.whl.metadata (7.7 kB)
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pipeline import Pipeline
from yang import util
from yang.chronos import Chronos

timeframe = "1h"
chronos = Chronos(timeframe=timeframe, lookback_periods=24 * 90)
pipeline = Pipeline(timeframe=timeframe)


25/01/08 10:00:32 WARN Utils: Your hostname, 0xglebs-MacBook-Pro.local resolves to a loopback address: 127.0.0.1; using 172.20.10.3 instead (on interface en0)
25/01/08 10:00:32 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/01/08 10:00:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
reload = False

candles_file_name = f"ohlcv{timeframe}"
candles_path = f"{util.DATA_DIR}/{candles_file_name}.csv"

if reload:
    candles_df = await pipeline.get_candles_df(timeframe=timeframe)  # noqa: PLE1142
    pipeline.save_csv(candles_file_name, candles_df)

candles_df = pipeline.spark.read.csv(candles_path, header=True, inferSchema=True)

candles_df.show()

+-------------------+--------+--------+--------+--------+--------+-----------------+-------+
|          timestamp|    open|    high|     low|   close|  volume|           symbol| ticker|
+-------------------+--------+--------+--------+--------+--------+-----------------+-------+
|2024-06-01 14:00:00| 0.69529| 0.69596| 0.69302| 0.69569| 44339.8|  MATIC/USDC:USDC|  MATIC|
|2024-06-01 14:00:00|  10.148|   10.16|  10.091|  10.091|  7967.2|   RNDR/USDC:USDC|   RNDR|
|2024-06-01 14:00:00|  13.534|  13.534|  13.534|  13.534|     0.0| UNIBOT/USDC:USDC| UNIBOT|
|2024-06-01 14:00:00| 0.01142| 0.01142| 0.01142| 0.01142|   955.0|     OX/USDC:USDC|     OX|
|2024-06-01 14:00:00| 0.00156| 0.00156| 0.00156| 0.00156|     0.0|   SHIA/USDC:USDC|   SHIA|
|2024-06-01 14:00:00| 0.25149| 0.25149| 0.24961| 0.24993| 14934.0|    BLZ/USDC:USDC|    BLZ|
|2024-06-01 14:00:00|0.082201|0.082318|0.081419|0.081754| 47240.0|   LOOM/USDC:USDC|   LOOM|
|2024-06-01 14:00:00| 0.16529| 0.16567| 0.16148| 0.16242|156114.0|  CA

In [4]:
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


In [5]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql import window as W

# Get oldest and latest timestamps
oldest = candles_df.select(F.min("timestamp")).first()[0]
latest = candles_df.select(F.max("timestamp")).first()[0]
logger.info("Data range: %s to %s", oldest, latest)


def with_forward_return(df: DataFrame) -> DataFrame:
    forward_window = 7 * 24
    forward_window = chronos.symbol_window.rowsBetween(1, forward_window)
    return (
        df.withColumn("forward_return", F.exp(F.sum("log_return").over(forward_window)) - 1)
        .withColumn(
            "price_zscore_fw_return_corr",
            F.corr(F.col("price_zscore"), F.col("forward_return")).over(chronos.rolling_window),
        )
        .withColumn(
            "mean_reversion_score",
            F.avg("price_zscore_fw_return_corr").over(
                W.Window.rowsBetween(-3, W.Window.currentRow)
            ),
        )
    )


return_df = (
    candles_df.transform(chronos.with_returns)
    .transform(chronos.with_sma)
    .transform(chronos.with_zscore)
    .transform(chronos.with_volatility)
    .transform(chronos.with_beta)
    .transform(with_forward_return)
)

return_df.describe().show()
return_df.printSchema()

INFO:__main__:Data range: 2024-06-01 14:00:00 to 2025-01-06 19:00:00            
INFO:yang.chronos:Calculating returns...
DEBUG:yang.chronos:Initial count: 1063978; Ticker count: 295                    
DEBUG:yang.chronos:Count column count (1063978) check passed.                   
DEBUG:yang.chronos:Return column count (1063681) check passed.                  
INFO:yang.chronos:Calculating SMA...
DEBUG:yang.chronos:SMA column count (514525) check passed.                      
DEBUG:yang.chronos:Mean return column count (514313) check passed.              
DEBUG:yang.chronos:Price stddev column count (514313) check passed.             
DEBUG:yang.chronos:Max/min column counts (514525, 514525) check passed.         
INFO:yang.chronos:Calculating zscore...
INFO:yang.chronos:Calculating volatility...
INFO:yang.chronos:Calculating beta...


+-------+------------------+------------------+-----------------+------------------+--------------------+---------------+-------+--------------------+--------------------+--------------------+--------------------+-------------------+-----------------+-----------------+------------------+-------------------+---------------------+--------------------+------------------+--------------------+--------------------+-------------------+-------------------+-------------------+---------------------------+--------------------+
|summary|              open|              high|              low|             close|              volume|         symbol| ticker|          log_return|                 sma|         mean_return|        price_stddev|      return_stddev|              max|              min|      price_zscore|             stddev|annualized_volatility|          btc_return|             count|          covariance|        btc_variance|               beta|     btc_cum_return|     forward_return|price_

25/01/08 14:53:21 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$driverEndpoint(BlockManagerMasterEndpoint.scala:123)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.isExecutorAlive$lzycompute$1(BlockManagerMasterEndpoint.scala:688)
	at org.apache.spark.storage.BlockManagerMasterE